In [1]:
#------------------------------------------------ Import Lib ----------------------------------------
import os
import re
import datetime
import requests
import pandas as pd
import pdfplumber
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from time import sleep
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'ME CBMONT' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__)) ## production environment (.py)
except NameError:
    scriptfolder = os.getcwd() ## notebook environment

os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running ME CBMONT Web Scraping Tool v.1.0


In [3]:
#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': []}

processdate = now.strftime('%Y-%m-%d')

# Lists from Jira DECD-6133 (ticket skips ListNr 3 and 4)
regdict = {
    1: {"ListName": "Register of Banks",
        "URL": "https://www.cbcg.me/en/core-functions/supervisory-function/banking-system/register-of-banks",
        "Comments": "each link on this page is an entity, click to open the entity profile page and extract the info (name, address, id)"},
    2: {"ListName": "Microcredit Financial Institutions (MFIs)",
        "URL": "https://www.cbcg.me/en/core-functions/supervisory-function/financial-service-providers/microcredit-financial-institutions-mfis",
        "Comments": "each link on this page is an entity, click to open the entity profile page and extract the info (name, address, id)"},
    5: {"ListName": "Leasing Companies",
        "URL": "https://www.cbcg.me/en/core-functions/supervisory-function/financial-service-providers/leasing-companies",
        "Comments": "NEW LIST! each link on this page is an entity, click to open the entity profile page and extract the info (name, address, id)"},
    6: {"ListName": "Factoring Companies",
        "URL": "https://www.cbcg.me/en/core-functions/supervisory-function/financial-service-providers/factoring-companies",
        "Comments": "NEW LIST! each link on this page is an entity, click to open the entity profile page and extract the info (name, address, id)"},
    7: {"ListName": "Companies for the Purchase of Receivables",
        "URL": "https://www.cbcg.me/en/core-functions/supervisory-function/financial-service-providers/companies-for-the-purchase-of-receivables",
        "Comments": "NEW LIST! each link on this page is an entity, click to open the entity profile page and extract the info (name, address, id)"},
    8: {"ListName": "Register of Payment Institutions",
        "URL": "https://www.cbcg.me/en/core-functions/payment-system/registers/register-of-payment-institutions",
        "Comments": "NEW LIST! each link on this page is an entity, click to open the entity profile pdf and extract the info (name, address, id, contact)"},
}

ListLabeldict = {1: 1, 2: 4, 5: 4, 6: 4, 7: 4, 8: 4}

BASE = 'https://www.cbcg.me'
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36'}

In [4]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def get_soup(url):
    r = requests.get(url, headers=HEADERS, timeout=60, verify=False)   # corporate proxy re-signs TLS
    r.raise_for_status()
    return BeautifulSoup(r.text, 'html.parser')

def add_row(rowdata):
    for key in sqldict:
        sqldict[key].append(rowdata.get(key, ''))

def common_fields(listnr):
    return {'ListLabel': ListLabeldict[listnr],
            'RegCtry': 'ME',
            'RegCode': 'CBMONT',
            'ListCode': str(listnr),
            'ListName': regdict[listnr]['ListName'],
            'ListLanguage': 'EN',
            'ListProcessDate': processdate,
            'RegulationType': 'Regulated',
            'Cntry': 'ME'}

def split_city(address):
    """profile addresses end ', Podgorica' - split the city off when the tail is letters-only"""
    m = re.search(r',\s*([A-Za-z\u010d\u0107\u0161\u017e\u0111\u010c\u0106\u0160\u017d\u0110 ]+)$', address.strip())
    if m:
        return address[:m.start()].strip(), m.group(1).strip()
    return address.strip(), ''

def parse_profile(soup):
    """profile pages: name = h1; labeled '<strong>Label</strong>: value' paragraphs.
    Label variants per entity type (banks use an en dash 'Head-office'):
    Head-office / MFI Head office / Leasing company head office / Factoring company head office / Head office
    Licence no / Licence No. / Licence No"""
    content = soup.select_one('main div.page-text')
    name = content.select_one('h1').get_text(strip=True)
    txt = re.sub(r'[ \t]+', ' ', content.get_text('\n', strip=True))
    ho = re.search(r'(?:Head[\s\u2013\u2014-]*office|head office)\s*:?\s*\n?\s*(.+)', txt, re.I)
    # date variants: '0101-72/1-2002 of 18 December 2002', '... od 18 of December 2002' (Montenegrin 'od')
    lic = re.search(r'Licen[cs]e\s+[Nn]o\.?\s*:?\s*\n?\s*([0-9][0-9/\u2013\u2014-]*)\s*(?:of|od)?\s*(\d{1,2}\s+(?:of\s+)?\w+\s+\d{4}|\d{1,2}\.\d{2}\.\d{4})?', txt)
    address = ho.group(1).strip() if ho else ''
    lic_no = lic.group(1).strip() if lic else ''
    lic_date = ''
    if lic and lic.group(2):
        raw = lic.group(2)
        try:
            if re.match(r'\d{1,2}\.\d{2}\.\d{4}$', raw):   # numeric variant: 'of 06.04.2015'
                lic_date = datetime.datetime.strptime(raw, '%d.%m.%Y').strftime('%Y-%m-%d')
            else:
                cleaned = re.sub(r'\bof\b', ' ', raw).split()
                lic_date = datetime.datetime.strptime(' '.join(cleaned), '%d %B %Y').strftime('%Y-%m-%d')
        except ValueError:
            lic_date = raw   # keep verbatim if unparseable
    return name, address, lic_no, lic_date

In [5]:
#------------------------------------------------ Begin_Main : Lists 1, 2, 5, 6, 7 - entity links -> profile pages ----------------------------------------
for listnr in [1, 2, 5, 6, 7]:
    print(f"[INFO] : Working _({regdict[listnr]['ListName']})_ ")
    soup = get_soup(regdict[listnr]['URL'])
    links = soup.select('div.children-listing a.children-item')
    count = 0
    for a in links:
        prof_url = urljoin(BASE, a['href'])
        prof = get_soup(prof_url)
        name, address, lic_no, lic_date = parse_profile(prof)
        addr1, city = split_city(address)
        row = common_fields(listnr)
        row.update({'Name': name, 'Address_1': addr1, 'City': city,
                    'InternalID_1': lic_no, 'InternalID_1_type': 'Licence No.' if lic_no else '',
                    'RegulationDate': lic_date})
        add_row(row)
        count += 1
        if not address or not lic_no:
            print(f"[WARN] : incomplete profile for '{name}' ({prof_url}) - address='{address}', licence='{lic_no}'")
        sleep(0.5)
    print(f"[INFO] : List {listnr} -> {count} entities")

[INFO] : Working _(Register of Banks)_ 


[INFO] : List 1 -> 11 entities
[INFO] : Working _(Microcredit Financial Institutions (MFIs))_ 


[INFO] : List 2 -> 12 entities
[INFO] : Working _(Leasing Companies)_ 


[INFO] : List 5 -> 1 entities
[INFO] : Working _(Factoring Companies)_ 


[INFO] : List 6 -> 2 entities
[INFO] : Working _(Companies for the Purchase of Receivables)_ 


[INFO] : List 7 -> 2 entities


In [6]:
#------------------------------------------------ Begin_Main : List 8 - Register of Payment Institutions (PDF profiles) ----------------------------------------
listnr = 8
print(f"[INFO] : Working _({regdict[listnr]['ListName']})_ ")

soup = get_soup(regdict[listnr]['URL'])
pdf_links = soup.select('main div.page-text div.table-responsive table a[href]')
count = 0
for a in pdf_links:
    pdf_url = urljoin(BASE, a['href'])
    local = os.path.join(tempfolder, os.path.basename(pdf_url))
    r = requests.get(pdf_url, headers=HEADERS, timeout=60, verify=False)
    r.raise_for_status()
    with open(local, 'wb') as f:
        f.write(r.content)

    with pdfplumber.open(local) as pdf:
        text = pdf.pages[0].extract_text()
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    i = next(k for k, l in enumerate(lines) if 'Registry of payment institutions' in l)
    name = lines[i + 1]
    addr_lines = []
    for l in lines[i + 2:]:
        if l.startswith('Registration number'):
            break
        addr_lines.append(l)
    zipc = city = ''
    if addr_lines:
        m = re.match(r'(\d{5})\s+(.*)', addr_lines[-1])   # last address line = 'ZIP CITY'
        if m:
            zipc, city = m.group(1), m.group(2)
            addr_lines = addr_lines[:-1]
    def grab(pat):
        m = re.search(pat, text)
        return m.group(1).strip() if m else ''
    reg_no = grab(r'Registration number of the payment institution:\s*(\S+)')
    # decision date may sit on the same line or 1-2 lines below the label ('approval: 09.03.2020')
    md = None
    for k, l in enumerate(lines):
        if 'Number and date of the Decision' in l:
            md = re.search(r'(\d{2})\.(\d{2})\.(\d{4})', ' '.join(lines[k:k + 3]))
            break
    row = common_fields(listnr)
    row.update({'Name': name, 'Address_1': ', '.join(addr_lines), 'Zip': zipc, 'City': city,
                'InternalID_1': reg_no, 'InternalID_1_type': 'Registration No.' if reg_no else '',
                'Phone': grab(r'Contact phone:\s*(.+)'),
                'Email': grab(r'E-mail address:\s*(.+)'),
                'Website': grab(r'Internet address:\s*(.+)'),
                'RegulationDate': f"{md.group(3)}-{md.group(2)}-{md.group(1)}" if md else ''})
    add_row(row)
    count += 1
    if not reg_no or not zipc:
        print(f"[WARN] : incomplete PDF parse for '{name}' ({pdf_url})")
    sleep(0.5)

print(f"[INFO] : List 8 -> {count} entities")

[INFO] : Working _(Register of Payment Institutions)_ 


[INFO] : List 8 -> 7 entities


In [7]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)

df = df[df['Name'] != '']

df.to_excel(os.path.join(scriptfolder, filename), sheet_name='SQL Ready', index=False)

print('Saved {} rows to {}'.format(len(df), os.path.join(scriptfolder, filename)))

Saved 35 rows to /Users/wuj1/Library/CloudStorage/OneDrive-Moody's/Desktop/Regulator/ME CBMONT/ME CBMONT SQL Ready 2026-07-13 10.36.58.xlsx
